# 📋 MNT02 미션 01 — 야전 정비창 수리부속 청구/출고 이력 및 장비 가동률 데이터셋 스키마 정의

**Solution 교관 전용** / 학생 공개 첨부 금지 / 교육용 가상 자료

---

## 시작 전 꼭 읽으세요

이번 활동은 **교육용 가상 부품 데이터**를 분석하는 실습입니다. 코딩이 처음이어도 안내와 TODO를 따라 한 단계씩 진행하세요.

1. 아래 **준비 셀**을 실행합니다. 필요한 라이브러리는 실습 환경에 설치되어 있어야 합니다.
2. 미션의 입력 자료·핵심 함수를 확인합니다. 학생용 `TODO` 부분을 직접 작성하거나 허용된 AI에 도움을 요청합니다.
3. 코드를 실행하고 실제 출력·오류를 기록합니다. 실행이 실패했다면 실패 내용을 그대로 남기고 수정 후 다시 실행하세요.
4. AI에게 보낸 **실제 질문과 받은 답변 원문**을 작성 기록에 붙여넣고, 코드·출력과 비교한 수정 이유를 작성합니다.
5. 마지막 기록 셀을 실행해서 JSON과 검토용 ZIP을 만들고 내려받습니다. 파일 생성만으로 학습 PASS나 사람의 승인이 되지 않습니다.

**인터넷/폐쇄망:** Colab과 외부 AI는 접속이 허용된 환경에서만 사용합니다.

폐쇄망에서는 사전 설치된 로컬 Jupyter 등 허용된 환경이 필요하며, 외부 AI 사용이 불가능하면 '미실시'로 기록하세요.

**보안:** 실제 부대·조달·업체·정비 자료를 입력하거나 AI로 실제 발주를 승인하지 마세요.

**실습 전 필수:** ① 환경 설정 셀만 실행하면 됩니다.

실습자료(미션 lesson·forms, 공통 안내)가 노트북에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.

준비 완료 문구를 확인한 뒤 다음 셀로 진행하세요.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'MNT02_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
MNT02_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global MNT02_PROJECT_READY, MNT02_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        members = [m for m in archive.infolist() if not m.is_dir()]
        if not members:
            raise ValueError('ZIP 안에 실습자료 파일이 없습니다.')
        if len(members) > 15000 or sum(m.file_size for m in members) > 800*1024*1024:
            raise ValueError('ZIP이 너무 크거나 파일 개수가 너무 많습니다.')
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    MNT02_PROJECT_DIR = WORKSPACE
    MNT02_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 4개: MNT02_Colab_실행안내.md, MNT02_M01_forms.md, MNT02_M01_lesson.md, MNT02_공통_작성안내.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAgIAAAAIVz+D1bE0QYAANcMAAAbAAAATU5UMDJfQ29sYWJf7Iuk7ZaJ7JWI64K0Lm1kbVdNTxtXFN3zK55UdVkH8lW6RCmqLAFFcbLpBpngSG7B'
    'tmyzSFcGBuRgU0ywwziMydCYzzrN4A88qGbjnzPvzX/oue8920PCAhkzb+4799xzz718x2bnXow/ZM+Sy9FFJgp1//1bJip5vt4eG5sIqacLmezqUiyRDcVT'
    'bxKL/KTHxEHba9lM1HP8xmBiMycOSsKw9HGEEdsdv2zwY0vYNX/bDf0ZT4kaTpoNfnrLC3leqIeYsA3RbAvbEkd7vOWy38LzXqvHvE6DXxt+5WJwznMdcdxm'
    'Im8Ku8L83TbfKfPtMhNHDYXYr7T5rsH4zr/ifVvsO+IshyS2AUKHGHsYYn7FEB/KfsVivNYTtTZ3i/wj4DT4WZEJoy5uzMCdidWV1Jt+NxVNLEUz/W7mVZy+'
    '4uOPePaH5Vg0neh3V6LZ1HIyuxxflLmtN8SHi0AIUXX5dY5IRYoA5+BS33D63an5MPPX6/2uODqR/B04XpOeDjGa32bwKMT4F1dsmmwmlskkcf3rZHolI2om'
    'EeabBvNahqjcSii1WyoPOBU1l04QNttgfrHEC+cAEEa9GD879lwXh3vyHbsSxK5AT4UZ8Ijm536XFzq8lQPm5j9MtPPQSL/rdXvelcN4MSeOkRdvtUXVwJGO'
    'Jf7eBKe2v1Gj2PTguD0KP/Z4kAxvGVoGbCn2mqQVX0ktx1agtmg2nkww3nDFUQl6I8TQiTCuUG2WjqWWo69iC69XE6/onLB6EBtC5cBpMA/U+ZQg1SgMELsO'
    'WPGcd6IGgi0bPN8jtNsy37dIYaHFGFiOUWiVAy+ZUvhHJr/8j5cswsebFVxgWPxyk6KCEZSdl4ZR55LZ8CCp2NJ0Op1MM7Gx5m9YqmuklKWOSQKivocajV4f'
    'e0LS7YhCAx/n6IFBuf1Dg6VXEwsr8UyGGKAaNk1RK7JhUzDICh1FtH5dAmKGIgxSvi57Tg6ncl7zFkrqofqSUEkF0gNPe/jEU8gsGPfG8DqlQFzUVQMkGMVz'
    'aRfGFd8+kXfVc0S+rtY9naq7XLapUsh1Dk0T7PUhM09DLLmaTa1mMw9meTPvm+4D/0NROOeklUx2CQ/RstmlWDpNyAmySqLr8tNevwv2Qr9niDoQAKUC5ahq'
    'fHcTVsX45Tnfz8u6OjlUDUBK/FORsvKcNXzRb6Jn+ZeczKpypKHLzpME6qzkRXh/t6oZkNHtwXFSuL0GE4HKQIzUxR1uvFbH3+roKsmCHzTgYQOQw/cJzEV7'
    'UE3eLAsLPcCbBrVaTRadH9cCTP4Ip/+RibM1Ye8B4Sa3TwKuPsjcrqDa6C8UHDd/9is2cnjkXbVJTHTCa+b8LTsoMviOZUM+Zl6swyFQIr9qgi6mrINo9Kvg'
    'xQVkaqyZ6annc9PPF8Jz8y9f4OISImi49xkJo8AFi0XCsy9npl5M/0zKhFSk6FQ2yvOG/mCJa+s+fx04nnJxWVdYxmkNtDmW+ETh9mSH4r2DrYAvT4K6SVT6'
    '0fj491Qxf/ctENEfuOMQlXeGnbTZagXTSlQhKdVNhKrqjh5JcekhKFmBPnihQXer3yiVr2vIqFEqEJx7LJVGzdyhjPx3eX55IecEBgRVAanQr7DGC2eY9+EN'
    'KlEm2XjOlSjYw0DAXDF4oTpI+CckPDFO8Uh2O672eSo++JIXq9ZRYwB8QD9kLnfCYK60hbkmtQUg+9bEuOcAxtGJ7Bo9M1su/BWAMSDJdbqW17yQcwaskHNS'
    'mfVMGEni2oJ80UJUDcrcOVfLAp3DUDbQvt/CoTlDUu80MMpkLutSReg3sg29YKy3IQ5qMtyHaIplgx4p6Y/qPTYxLueJnMG5iXE9uwcqyJskpoA6Dku0AWkq'
    'Zx/3u7NP8TOpJjPJQ/d0ZHpm+hmEvjAbjkTCv85FSEcUXEbUDScFpPUhBQSnF1s72rugbBNdJPEbw/Yb8vdsNiL3kLz5QMPRpaIdaaAsmbN6NZgzNkZ+lifv'
    'OnsrDQSH9fBSQ4EGJcWUo1NTopZOhZfEKdtW8UnmAr+WNgDRw7gpXb9YxNLCKAfV6ncsZBU7WXQJI4F20cHiiR2P738OzL5BidXs1bsOFhn0QEBgENMFHiNZ'
    'eFelSlutlDObn4pEAKxT8tpQ0vaN3LMM7Rn3+4uepMPYtIbJujfkq8F1We+4cknTjge63tVJgNyuc+cwYEBjSjtgbiPPfolnmf+Xq7X5cDL0BGbkOUZAaqiv'
    'bLGtIi0T6rFenOQ8uZaDh+9foRxyyfhGIQPLQAQshyr+SBXKN4KCkv9b8I+Ov4FLDNoeUWwQLbtBDi21tanVSgXX/5CgxzFytrErS1lqgxo5TdmRtn5m3N0M'
    '/gdQSwMEFAAAAAgAAAAhXKUYyg3OBwAAfBIAABIAAABNTlQwMl9NMDFfZm9ybXMubWTlWE1PG1kW3fMrrpRNWiIxTc+KTStqjTSRJmx6dq0RQkB3o0lwBGik'
    'llgYeI4c7ChO2hUKUkblCQSYcZTC2KaQYBb8lPSu3qv/0Ofe955tmnRGGk3PphUEdr2v++4959xTuUUPpv8yMUkPJj4ns/fCqBMyr16YapM+lBpkggsTKzJx'
    'oM+VSS7JVEL9tq37JfPkGZnOu6zXLph+lJ3GZJpdHR+QTp5jowPMpywp6ec7+iAm/SzBcK4SU66S2drPN/b1YYX3Nc1wbMy8ruvTlPLnT7Fkir7JA4WD/jpc'
    'dmMN2fC2y7rdpbyhdCsyKsUs0oe9wUV2lH5TGydsp3+MaHIiSyIO02yuu23G5S9/3QzzRo2yTsm8bZIJ9/VBSBPZSRcz3lyYvbJ/hnTg9jKxH2EwrJi0RbpW'
    'wi63EbTZbXyWnV6Qrh6Zf2BVpPLtkHRfGdUk/c/j/EWEO+JTOd8J8oC/3B0b06elrHNJWGjiaIru3Tc7JVooPpxZKq4uzi3MzM+uLrhrXJ3pfoUzu4Vcb8Tm'
    'EE+mZ6cLSAS2QyBXZ2avjnvnYeCC1gcX7mZZmpjtOpl+AzvkQSh1kwwUrl8z6yQIhheikFmvzltX9xHprVvDWuAkmW33HhtbI93um2bt6sxlfI0GoeDZdsXs'
    'HutOAzmRGcgWz3DHu/TaKNZcgl0cOAgIaad4fu8+RvInMR71Is5wXqsj17SG0+/cuUO/+hvjdO3nt3nACbp3n5DofPfZ1RkQpxNUyAELj3ULmbpDZqPNzHLR'
    'e8BWIxPXp+jDqxZvoqs9AIP0aRfrGHky4J6a7a7eq1H+WrnV2BS4QXX1YStLU0cOBhJDPoqn+NQYVO3lQewKgyxeNgRUGDHBJSMXqQYy9FbDJZ/XgZ2miZ+u'
    'AH5zHXBwo7I/anm4Dkwf6YMuL8Tl8uDdlCCiVsp3Gox8E69zuav7OIpAE6gHubLqsC7L4hKUgzljzsPhXH/hT871EYJrxw1E8ilAjH2ihhAjCQ00k2NNszSI'
    'WQIwTcUVMS3Q7VhXK0wLslkkR2MXoq/5kEpJaELJHW+wo1wW9HPlgDHckVMHBBi1D6UY3NmeYlScbzYH5ELtqvv5q6fgVLnG3NxKKTu7yE6Sa2mxSJMCCxx4'
    'fQT5YL2AbMVgKKJiDmLAUsvGi5U102QOulz/Fkwb+wJa67RZMgxBbSWjGZa7ku5U8jDlZJrk3zbJm4z57CxFbLgN0F9nVcuDrq6eQyG7EB1uTRuhr2N2qsxG'
    '4tM2Uh8g1Gz1Ripg6ylgR322DpDiWk03L/T7lJuSnyDU9ETSHcVfB2NC0F+O4bc8H41gBPc2/a7Ovlz/g7SLettey51gvyTHutMgAP2Ib23bMDL4DqxlKbZT'
    'k5fcehS4BTlLkyxRQ4p6mR7irtqGJGHB+xQfOW91QIxs65LkCAH+P6rNwodLdiuDQrivzLBuxQQVx2hXAlFKX7JyCkLp/rrQA8gyiuX8LNVvcV/YDnx0uRtZ'
    'DQChDxJub8qhiS4wj7QKWZZxfyHbUFzRXrOuWAr0NWJ13YxsDwALWPnl46hiyBOzdQ5q/1Rq6CpDHQTBvZoX6CG+Dj+VAil5HvRMtY1z2nmoBj0IhQ6OXA/e'
    'K8O6YXk/YgsHZ4Kd+XnnX+TFxYoEjcB2qL1JlHXaTBPbxQeTPRw+KcV/n324KP5mzf6DG3qfulj5pNqRaJ86YcczwKL0IriM6s5AIThYdYL+5gIUcIvTucZF'
    'GRONcc5mqLmcq52IPespJh4qufgvWveam0JrPk82phEyyK2HeulKKgD6D6DHyOf4zGYcKgUHMfVfWHAY3pcVUBehn/ay9Kk+bN72dlQiAcNRraPPxm8adSuz'
    '3C8GLk86/MeN9viIQ9eNwJQbWlU4OBFTNtjON7JEs+PbbQhKHNysinCnPRJPyJ5o95icubVmiIvoEYL9sNSBlFU8uOQ62ibp3Oyg4VmDZbve1RmcEaqxp3w5'
    'bXZ5teu8rypAGU7+0oFwjSZ9IQSKU7SyPFeYLz6aXVwqPFpanZicsV/uPv6B0+VjxNWtNbHx2ChBUqY9v2dIvdyhtLww543ghycvvfdmk7FXh79j14Rpd51t'
    'o6+/+tMfH9zjesx9vzD3N8mypAzNjpyh9i8JN4y3cEaJKWF3FlnOfMx8DxsiQoXiu8lQE2xg+semdeIMIihnGedFwuohjnwRFaxGYolzS8MMM2Jeqz8vrKwU'
    'l7hSwlFLUT5BWGT1F7hxpulmfb7w9ZGC/v7eGp3cDEjkWjNsEwKwEnwe4oa29fqXPWl8BavPAyl0SjjMsUPr9NfTBAiZalN4JnbRb+Q8sbOkzZA7PXus7bLp'
    'dAWamxVue4hHAAc0qzds0EywNXRZPpOTnDUSxR7Fovh5yRIrsk2foNgBc2APXeIYiocl7oIcr3/L1cddeTn5Mb5eODHeyYmpxggPOksjG7I14BAOCtaGsT/F'
    'plbbGebuLdRqOvOtbdP5EaT+wSHVImuK5otzK05DHha/W1zBK/6KU5OZlccLc3cfzY9DGx4Xl1f9PEjL7Mz84tzqYnFpdvkHmfFtcfnRSkH++2YG28/Id4ww'
    'pSyHBND+TdkaOHkds4y0/QnDckFrM7nw2wkUQNdDfw3v4djNpC37pgCJUhE/8EfYA8WSWNciQhgwXPcqgC/U/EZmfgZQSwMEFAAAAAgAAAAhXFXREQGGCQAA'
    'HxkAABMAAABNTlQwMl9NMDFfbGVzc29uLm1k5VjfTxvZFX73X3GlvOxKXgxkq1a8rKI0bSMVVDV9alUBBdqiJjgCVGnfjBkjB5NisjaMyQw1jfm18WqNGWMj'
    'mRf/KZu3uXf+h37n3HvH44TdrvrW5iXE8+Pec77zne98d+6J6ZnfjU+K6fEJ8V2uIlS1r+qOUPWqvHFU61aooitPm/I6p7ZeCtX+Juw0M+raC6/qQvmBrJ8I'
    '2doV6ugEz4uwlZO7NXlSF/JlC7cjp6UKJaG2G1G+Ic+KtK7y3VTq3j2hNjei2kV41Rfy64toz0ul8DYuCuwVvSoOV8AFRzm+kKVAlm7UUVkee8rxKDJ1WBl0'
    'Zelc/avA6216g646alJwYbcfXrbkSd+8HlUvZKkoS42x/yLHtEBECHPQDa86Ye+FPPM/GdlWRLvNqHr+afpDJCjgNzvIegiDquWEOijIZiCiisPp9HBTyLNO'
    'Opl3paoKFekUKTgkosGLqh5loTF8XZZXPYo/2n8hZLuiPIcWj3ZfIIBU6jP7RFR18JCQ3/ZUwQUCdUe1A6FKDbXdUQdlwCm3K2adIVLx+0B1syh+ubwuon/0'
    'lHMpJsZ+okpe2HLe5Sq0Su84bL1C2MDWkSX3Xa463FuvOmXwFmprB2UDazy16VOtkanGnMDj2kU1JIpN5XURZcDl45YsNQVew4NAyCDvOdGBK8J2Ltqqp5OI'
    'n+2FV054s6POhjj7ZXnqKN+ht3FXXhHuHNNp00BqS6+3HXR1zTNxyVH94R5OT8jmtfJ3lNenlYCw/MoTk+NhyyPWgBe0m62ZS3nNPJkZdH/x5OGgO/P48Yzy'
    'cyLsfCNbFwL0Vw361aIqqdsKLRW5VVXyOeSvz1WeohS2RdrFyO3RtnKvwqGbG2rTjSo79A5gUad+2GvpNMsEk45Crwt0mS5I86CoDi9QP1wqOtK/xd9CH7AI'
    '5Tbkics95LxR+wEveu3Jsgu2ELi6UJaRn4k/aJ79MUHi93tfaJm5m/0gwB7RS9Uc9Ez6LlDNMmn+G2ds0jURi/HwMsATb/rqqGCvgfNgm0kBN90iWCvkDuH2'
    'idaST1mNRthltYclCiDjfwVkjZTxYyxB8nyLkGn2p8RidmEt82xlfXxy9mn2L8tr68sLa7OL2Wfzyyuza8+XFsaeLbJm3t3ouq/rEJ+OkQ5qY/yKtobwVIsy'
    'Hwy6D7NP5/9EXUzdby/qphay0Iu2e/J6w14xWNd9XBa/f/wbDp6ZNiXm1lYXMjpEE7n+Mfb8y7kxVKGj0H7QNxJKsAFNErlgjqrnIHeGVqr91oYSvXbEg8es'
    'ctcVBu0qF7ZvR4Qlzn5KjEQ2tzi/Pp/57aMHP59+BKTmmH3nEGws5t8iD7PE/Z8acqAzUNpBd+Jn9sLrMrTJgDfoTk4QedBC8iXKdFSAnHN/dCD4zqjWlTwA'
    'LNT+HvoDoPBknMVknP1zdvXZGkXDheMCry49z66urxGjSVVrDhdG1ycqgRq6kBgf7aLRXXrWYKkRpMpGO2jX/g+XgB5j8PHY3+efLgOipTlmhaOqt1Nx87N8'
    'Ce55X6wuLRhsxHdbr2wXyGYPwKiDgCiNx8Ys1548/NWj6QdUs4W/Li38jecN3j68iKqBHR8aYBqxBC10JCEeVnPyQzXlBs3Y9tPNGLZbaDJDIi6A1Q7ZciXi'
    'SCjHps9gatrReDJdwaSlSpp5Pj1ByOqw48cMKy1cQ9qOlHykHMxbprFzyRsxm1mArwJ1HJglgS9+jFIbpQBFPUbNDNGAFIdWxNALO+Wk/0A/1j10CKlPVCML'
    'c9WDjhrPYiYaYDbIQZvqVZY/r87tkD8PbzZsEkSzqFqjno9qHsYTXbln/q9NHXgSHb4kcTOEiHbKkLlUyoiXVjeMNzCHtIwBtmXmnPxbbnFWSXBkRCN5zO0H'
    'cr9hCEITx1a3jjH8BS2ht0ygAIE4KEP0j8NeT9hZZJsW+IDZoBcPGHDjQrarZp4mCpjIdJIzNaqjB6guTSr1Pfr1oW4NyWkLjkVeNcyqQAcoGu3A+xS+qm9w'
    'i8RZjdBhyry5lH06u5LFHFiaJS7GfQSjQY5iGx46X1dnuDIzP5PBYKSu2MklGi0xiuPBzvszCW0mpuFGxl4iJ8ND733o7muSaPLRQDxuMYpHTY0LeGIoyLhq'
    'K5pUfvtgEgdOKPYwkJuzI/KGu2TAmqzBAekkJh7Z9Lyrim+RLZIBBFTy0YSvvWi/iObG/S+ocZTTwCAGQLrSyqlHm/59TP2k9bJc4th4tHNuJmoqcY2i6aBm'
    'Oq4GzQ3rae4i2efavHQ8Yr9mtO4qM/zjjtIGnKWikaP+ZlZoQbBR68H8vhAEiI3sOIPsUeIJUL/tATw8hXHSO0ZF4iXU9g2DxoeRpLrG75oZAQuaMJYQFNg4'
    'W0azlIHUd2m7vEteDYcFDmizyG4S/CCZD9gWkseubg8HszVmk2TCBMOTnAA8MNl0wYKb2t41j60PgxBjC4xIjtfSXV4EPNK/qo/6QKpj2LpUpTrCwzlPjNZR'
    '0FWHjLyxiwjJB7XL8eHzI8dHU46DYgtuGExGZe5H+FoyJXPGGVkXAzs3u7i8sL6cXZlf/dI8xIbK2CX0SzoxE/UpD8z299Cbae62oIjfSQ1KW8kG3mEXR0X8'
    'I0/7NDgSMzfpz3HQ++cLk45GlnSRdA6wAiqefjjQ8iEPT+sDhnMJhTedbhSSFsXaYZDjM72OW4/sRP/R+XKXj5jajo10ose9ygAzbbh57TkSR8ESmt79gDjG'
    'UbL4sGFwjSpROmZQ2nJNjPEXHT34p/4/vnjQvCMjelihuayd/LBqFBw7CDatwsxVsvQ4+4LP1ghSv6PLINf1E2OguXN4oMSjgewYPcl2SZd/0MUw9xx15Fj5'
    '0Ohyc+oaxyMqNWnwZ6MwJb7f2xNKCY+anKs6uHe5yv++vX+XI/Omri/U8SWpE7n09ltts3Ujo4Cmk9EAGd3NeMX4sOHsJ6K8dn69tLaWXaECseDp8Uo7cDea'
    'A8JBGUcK5W4kynLflIXLN/XRfa4wahW3jBE+zCsEoBX2xkWGIDz7bD32WH8zWn7jI5Exk8PCfPTzk2cXhXCSGdpRLGo+RIPd5linjSd/qDa2eUjQzzVBNaF+'
    '1KektPhP8zatp23mw88Z1EC6Y5jHCImHi/4UyI5U9x/J3o2L25yXPohSvQ9a6Hf9NZA5cIc/JUGCm8UFu4XekEce9iA1I9mrEkuPimb6DQFJJT5A/+D354xe'
    'Un+Gjr9Cj5nTuj6P8Syhs6DMByRrfJEIoA+62xXL19gT/RtQSwMEFAAACAgAAAAhXLEHCSjoBgAAnw0AABwAAABNTlQwMl/qs7XthrVf7J6R7ISx7JWI64K0'
    'Lm1kbVZdTxtHFH33rxgp75iQj6aPKKEtUkBRPhSpLxFNaYJEcQREVfu0hjUytlNM8MZrsussjY2BLupiG2wU8+Kf45n9Dz13ZtZePiQ+LO/OnXPPPffce4vN'
    'zD4fn2CD1mm4ccpEdVuYJ0xYWb7WTiQGgSHW00xkbb7v8zNDbHxg/EMg3HZoBnzPoSdityTsrOju9Tui6g9aHuNnpjDdfofnD8Q/GYYI4brDBk0j3PDwUu5c'
    'uF026CKAK1yTiXxN5E5Dy+a5Uiy6DBLaFgtLJq4SnhvmuqKa4fksz9fGmPhc5K2ucG0Wfszyo8N+Z/bZbL/zw7OHuAOwzm2JAClQ3MFpUTh7YveQqZzigXC/'
    '5wC1wQtGvzNomcK6QIxyRjTblFSdnwNltci/Fni9x+ioVwde0TDAVA7gdahE4tYtNjnNwoqDmyQ0ywQWsNsTnpVI3B5ji/MrK6klXNMNRA3nL0p8x5Fh3QtJ'
    'HuWEnzaiWPgUWmB5zQdIQHFt7nf7HVwhHFNUTcabJXxCtKYh9l3Gg0PetIhUfoAbEd5Lj+Axnj/lLSP2mLdrwunxQ1y27Yi8E1qHUSoTYyyCmK+FnzZRvwBp'
    'EFLeaou9NgggvJPTolzU7PQ7oQ1Yvs4KgD1Hcfm5GNW3ZQyaF7Ls3uUbozqQnprHONP8NwYYaFGIGL47Y1J8nhWjMCwUhNsjPDywhZ0OrTaQ+SqB2E3E39ce'
    '4uEWeZgJ0wvXXXZncIITbpsE4ppAHeVFolgLkF+mIHI1EmKF1CL2ghiku4BUtfnRN15UcoKEoQGiq2LqvCT5wxKrdtNM6nRiRQIheWeogxFRjgfpKZV7FlTe'
    'avO6DUTi1KGGU8lLiZzbsmBrB4PzdATzXtQ7INXiXzZ1nRFPFUAVGmH/6/K8L/+ju+VN1DvyRrzpFhg9ydhMlAOcIcSqqWNQX779M/lT6o/ky7dzqyi/ypxe'
    'C3c/jC4MHB44yaFBqCzCQpEMBARC2YOTYNSx5EtpXXZJcOQZsBoXfAoXyt7K8P0LrV0pIMekVho01/EuPww0eCpz1hGmg2qSy2iptOGHoBn9W0H/KAej5CGH'
    'rYrwTBDDiyTowakPEeCDsPBlZdhpUuYjZpXKeNPEWepW2Uq5kg5fLZJLSU41qnJRNNLC21bUxmuvKyf1OnO335m5j98HfIsiw6/IKsBFuk0hkBWAZe1L6pcm'
    'pRAoh2rDZxMJFRhQ17Psx4VVFv7d1VROPBi7Bx0OAvOa+zJyfnr0kXhEAt09+gxL2XM4GFzzb7JJ2KEnGzfvgBWkjj7Iik9tJMHRvihPt8C/+Bg5UiJkPmHF'
    'JtsoF3kpwDu4hB5JTHgdxcC5RpYaApFihnd0CGNjSsnIZdDpqjpfEyqio8cxJnzElN5U6QIjeo9QgUXJLht9Ve9R/+e9m+cAyIVkk9T6npVU1UqGW5uULO7G'
    'HSSAM5MfHVCko8wVK4RYyxSXJi/aRg1SurnukT2R1rIQMQYTBg36sWCQ1CQhLNw9UOrS01s1bax2Wml5X1uuHuJwNN00u20yzyjMsP9vnJkYviy0uygQ5EKa'
    'Kme4D6Ia27LxDDXd9aDiwad4G0OIahvAOcn5mRMJceY72fmy30itsHFAeTY98+Lx5POpR1p4cUS6X45Dy9MjGmVCPa+0GcXyaERGY9x5PDX5dHbq6avp2Scv'
    'nkfaNenwVZFQHbIYFuoSjGKEFOU2CWu0XuDGM1N5TDTR0JNYB6oYk7D9fTeq4MgtoIIdoD7DhHAomspXFdWFF3mT06RBxQbNA8jwpsaCjWKwDS2UsDRqUh4Q'
    'L1UKMqZKuaQIEk+1DuS6CDS9dR31sib79/hyLrKuQ8s3RHkjrnu9y8U2tp+nn4Dsh6nFuV/Ikmyfe7Xwo4GceKaL5/wsrbzj8ojGxMTgDT+bHG3dMHhjU2QM'
    '5Wnv3y2m5n5dScrFdeyvhXc4zxsFvnM8PAzEl6uuRiWKHW0mVDzTkdSElVLkM8MdBxWRJJEkud+L6WxhaWV1+f3r1dQyE0GXnEyPGAOv0gFIAmsa0v/mD8cP'
    'ETiEg1qOLqUTkn1KImYo1/dKitJEBWVpo81XmjR1j5o0iUSc+qGFxxxcyhubPvakONtQpEfLCrUGqJLv6uO0wmURRK126p5rCtaZbBTISORp4jEPaz4nQnSG'
    'huqreNd6DiYt5o3E7eOv2q00tWrnzpVWUu+XX8+/Si0vvFlYmltcSb5ZWH21PP/b/PL8Er6fGJ+4f3t8/HvSxyX/J7LWsfgcPJ2afDQzhcAqi4Xf36WWV2no'
    '7pygxXVSV6ebWt61e1OB5d5Fi2zRvjptXeP2OO0x9d7VecuiL5xo47Dv4E01DNBemFFUe1jAdvHqqpf4H1BLAQIUAxQAAAgIAAAAIVz+D1bE0QYAANcMAAAb'
    'AAAAAAAAAAAAAACkAQAAAABNTlQwMl9Db2xhYl/si6TtlonslYjrgrQubWRQSwECFAMUAAAACAAAACFcpRjKDc4HAAB8EgAAEgAAAAAAAAAAAAAApAEKBwAA'
    'TU5UMDJfTTAxX2Zvcm1zLm1kUEsBAhQDFAAAAAgAAAAhXFXREQGGCQAAHxkAABMAAAAAAAAAAAAAAKQBCA8AAE1OVDAyX00wMV9sZXNzb24ubWRQSwECFAMU'
    'AAAICAAAACFcsQcJKOgGAACfDQAAHAAAAAAAAAAAAAAApAG/GAAATU5UMDJf6rO17Ya1X+yekeyEseyViOuCtC5tZFBLBQYAAAAABAAEABQBAADhHwAAAAA='
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 상황: 어떤 문제를 해결하나요?

**미션 개요**

야전 정비창의 원활한 군수 지원과 단종 위험 분석을 위해 수리부속 청구/출고 이력 및 장비 가동률 데이터의 기초가 되는 데이터셋 스키마와 업무 프로세스를 정의하는 실습입니다.

**배경 및 필요성**

장비 정비 과정에서 발생하는 수리부속 청구와 가동률 데이터를 체계적으로 관리해야만 향후 수요 예측 및 단종 위험 분석의 정확도를 확보할 수 있습니다.

데이터 수집 초기 단계에서부터 표준화된 스키마와 철저한 정합성 검증 체계를 마련하여 오류 데이터 유입을 원천 차단할 필요가 있습니다.

**핵심 과업**

- 업무 프로세스 맵 정의: 정비 소요 발생부터 수리부속 청구, 재고 확인 및 불출, 조달 요청, 단종 위험 검토, 장비 가동률 집계에 이르는 전체 흐름을 단계별로 파악하고 정립합니다.
- 데이터 스키마 및 딕셔너리 구축: 청구/출고, 품목·공급망(단종 상태 포함), 장비 가동률 등 3개 묶음을 중심으로 20개 이상의 필수 필드 스키마와 데이터 딕셔너리를 정의합니다.
- 국방 표준 군수 코드 형식을 모사한 가상 번호(NSN, FSC, NIIN 등)를 적용합니다.
- 정합성 검증 및 오류 검출 시험: 가상 샘플 데이터를 생성하여 검증기를 통해 점검하고, 형식·범위·허용값·필수 필드 누락 및 NSN 논리 불일치 등 고의로 주입된 오류를 정상적으로 탐지하는지 시험합니다.

**검증 및 산출물**

**검증 기준:** 필드 20개 이상 정의, 정상 샘플 검증 오류 0건, 주입된 오류 6종(형식·최소값·누락·허용값·최대값·NSN 논리 불일치) 모두 완벽 검출 여부 확인.

**주요 산출물:**

docs/mnt02_logistics_domain_spec.md 및 데이터 딕셔너리, 스키마·딕셔너리 정의 코드.

## 📁 이번 미션에서 완성할 것

- **미션에서 완성할 내용**
  - 정비 소요 발생부터 청구, 재고 확인·불출, 조달 요청, 단종 위험 검토, 가동률 집계까지 업무 흐름을 단계별로 정리한다.
  - 청구/불출·품목/공급망·가동률 세 묶음으로 필드 20개 이상을 정의하고 NSN·FSC·NIIN은 국방 표준 군수 코드 형식을 모사한 가상 번호로 둔다.
  - 가상 샘플을 검증기로 점검하고 형식·범위·허용값·누락·논리 오류를 넣어 검출되는지 확인한다.
- **구현 계약**
  - 핵심 구현 계약 src/domain/mnt02_domain.py / validate: 가상 청구 기록 rec 사전 → 오류 문자열 목록.
  - 제공 SCHEMA와 check를 사용해 필수 필드·자료형·허용값을 검사한다.
  - 정상/오류 주입 결과를 구분한다.
- **변경 과제와 검증**
  - 추가 변경·검증 야전 정비창 수리부속 청구/출고 이력, 품목·공급망(단종 상태 포함), 장비 가동률 자료의 스키마와 업무 프로세스 맵, 데이터 딕셔너리를 정의한다.
  - AI와 eol_notice_date 필드·불가능 날짜·NaN/무한대·자료형 오류를 검증기에 추가하고 정상/주입 오류 결과를 비교한다.
  - 핵심구현 후첫성공실행을보존하고추가변경·재실행·실제AI원문·근거검토3건·보완·최종판단을 작성합니다.

**제출 경로:** `docs/mnt02_logistics_domain_spec.md`, `reports/mnt02_data_dictionary.md`, `forms/MNT02_M01_forms.md`


## 🔍 교관 검토 기준

### 정량 평가와 측정 근거

- [정량적 평가] [필수] 데이터 스키마 정의 — 업무 프로세스 맵 작성 완료, 필드 20개 이상 정의, 정상 샘플 검증 오류 0건, 주입 오류 전부 검출, 예측 대상(수요)과 단종 위험 분석 목표가 명확한가.
- PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다.
- 목표미달·0기준·미측정은그대로표시합니다.
- 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.

### 정성 평가와 검증 근거

- [정성적 평가] AI 활용: 본인 계획·실제 대화·코드 변경·검토3건·보완·재실행·전후 비교·최종 선택.
- PASS 증거 가상 NSN 형식을 유지하고 실제 코드 의미나 업체를 새로 조사해 넣지 않습니다.
- 정상 20건 과 오류 주입의 대상별 검출을 구분합니다.
- 단종 공지일을 추가하면 모든 정상 샘플도 갱신해야 합니다.
- 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.

### 진행 추적 및 완료 기준

- [진행 추적 및 완료 기준] 정성·검증: 가상 NSN 형식을 유지하고 실제 코드 의미나 업체를 새로 조사해 넣지 않습니다.
- 정상 20건 과 오류 주입의 대상별 검출을 구분합니다.
- 단종 공지일을 추가하면 모든 정상 샘플도 갱신해야 합니다.

### 제출 파일 및 완료 기록

- 파일: docs/mnt02_logistics_domain_spec.md, reports/mnt02_data_dictionary.md, forms/MNT02_M01_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
- 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
- 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

> **주의:** 아래 구현 코드는 원본 교관용 Workbook의 참고 구현입니다. 실행 결과와 기준 달성 여부는 실제 환경에서 독립 검증해야 합니다. 승인·발주·실측 사실을 생성한 것이 아닙니다.


## 📋 순서대로 할 활동

### 활동 1: 계획과 사전 판단

- 원본 목표 및 기본 자료 분석 야전 정비창의 원활한 군수 지원을 위한 원본 목표와 기본 제공 코드를 철저히 분석합니다.
- 수리부속 청구/출고 이력, 품목 및 공급망(단종 상태 포함), 장비 가동률 등 핵심 도메인 데이터를 어떤 필드로 각각 구분하고 구조화해야 하는지 검토합니다.
- 수요 및 단종 위험 분석 범위 설정 향후 월별 품목 수요 예측과 결품·과잉재고 방지, 그리고 단종예고·단종 품목의 최종 구매 수량 판단에 필요한 핵심 변수들을 식별합니다.
- 분석 과정에서 AI에 위임할 범위, 활용할 자료, 기술적 제약 사항, 그리고 산출물의 정합성을 검증할 구체적인 방법을 사전에 기획합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M01 계획: 야전 정비창 수리부속 청구/출고 이력, 품목·공급망(단종 상태 포함), 장비 가동률 자료의 스키마와 업무 프로세스 맵, 데이터 딕셔너리를 정의한다.에 필요한 입력·기간·단위·사용 가능 시점과 validate의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 2: AI와 코드 변경

- **함수 구현과 첫 실행**
  - 핵심 함수(validate) 구현 및 첫 실행 보존 가상 청구 기록(rec 사전)을 입력받아 오류 문자열 목록을 반환하는 핵심 검증 함수 validate를 구현하고 저장한 뒤, 첫 성공 실행 결과를 온전히 보존합니다.
  - 제공되는 SCHEMA와 check 함수를 활용하여 필수 필드 누락, 자료형 오류, 허용값 위반 여부를 철저히 검사합니다.
  - 정상 데이터 및 오류 주입 결과 구분 검증 정상 샘플 데이터가 검증기를 통과하는지 확인하는 동시에, 형식 오류, 최소/최대 범위 초과, 필수 필드 누락, NSN 논리 불일치 등의 오류를 고의로 주입하여 결과가 명확히 구분되는지 검증합니다.
- **AI 활용과 변경 과제**
  - AI 협업 및 변경 과제 수행 AI와의 협업을 통해 기획 내용을 실제 코드 구현에 체계적으로 반영합니다.
  - 변경 과제: AI와 함께 eol_notice_date(단종 공지 일자) 필드를 추가하고, 불가능한 날짜, NaN/무한대 값, 그리고 자료형 오류 등을 검증기에 새롭게 추가하여 정상 및 주입 오류 결과의 변화를 비교·분석합니다.
  - 실제 코드와 실행 출력 결과물은 모두 안전하게 저장합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M01 구현: src/domain/mnt02_domain.py의 validate를저장하고계약〔가상 청구 기록 rec 사전 → 오류 문자열 목록.
- 제공 SCHEMA와 check를 사용해 필수 필드·자료형·허용값을 검사한다.
- 정상/오류 주입 결과를 구분한다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 3: 근거 대조와 재실행

- 수정 코드 저장 및 재실행 검증 도메인 스키마와 검증기에 반영된 수정 코드를 안전하게 저장하고 스크립트를 재실행합니다.
- 단순히 날짜 형식이 일치하는지 여부를 넘어, 유효하지 않은 달력 일자나 숫자가 NaN 또는 무한대 값으로 입력되는 비정상 데이터 오류를 검증기가 정확히 탐지하는지 확인합니다.
- AI 설명 및 코드 선택 대조 분석 AI가 제시한 설명 및 코드 선택 사항 3건 이상을 실제 실습 자료와 실행 결과 로그에 직접 대조하여 타당성을 엄격히 검토합니다.
- 보완이 필요한 답변과 수정 사항을 다시 한번 확인하고, 가상 NSN 형식을 철저히 유지하며 임의로 실제 업체를 조사하거나 코드를 변경하지 않도록 유의합니다.

#### 확인할 사항:

- **판정에 필요한 근거**
  - 이 활동의 확인 근거 M01 검증: [필수] 데이터 스키마 정의 — 업무 프로세스 맵 작성 완료, 필드 20개 이상 정의, 정상 샘플 검증 오류 0건, 주입 오류 전부 검출, 예측 대상(수요)과 단종 위험 분석 목표가 명확한가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?
- **미션별 확인 사항**
  - 가상 NSN 형식을 유지하고 실제 코드 의미나 업체를 새로 조사해 넣지 않습니다.
  - 정상 20건 과 오류 주입의 대상별 검출을 구분합니다.
  - 단종 공지일을 추가하면 모든 정상 샘플도 갱신해야 합니다.
- **AI 검토 기록**
  - AI 주장/선택3건을근거로검토·보완·재실행했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 4: 최종 판단과 산출물

- **산출물 제출과 상태 기록**
  - 종합 산출물 제출 및 상태 구분 원본 평가 기준, 변경 전후 비교 코드, 실제 AI 대화 내역, 그리고 최종 보고서를 누락 없이 체계적으로 제출합니다.
  - 미달 항목, 미측정 항목, 그리고 실제 승인 대기 상태를 명확히 구분하여 기록합니다.
- **미션별 검증 원칙**
  - 가상 규격 및 검증 기준 준수 국방 표준 군수 코드를 모사한 가상 NSN 형식을 엄격히 유지하며, 실제 업체의 정보나 외부 코드를 임의로 조사하여 혼입하지 않습니다.
  - 정상 데이터 20건과 고의로 주입된 오류 데이터의 대상별 검출 결과를 철저히 분리하여 검증합니다.
  - 단종 공지일(eol_notice_date) 필드가 추가됨에 따라 모든 정상 샘플 데이터 역시 이에 맞춰 갱신되었는지 최종 점검합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M01 완료: docs/mnt02_logistics_domain_spec.md, reports/mnt02_data_dictionary.md, forms/MNT02_M01_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

## 이 미션의 AI 활용 예시

- 가상 자료 분석·기획·코드 변경·디버깅·검증 설계·해석
- "교육용 미션 01의 핵심 함수 구현에서 누락되기 쉬운 입력값·경계조건을 알려주세요. 원본 자료에 없는 성능 수치를 만들지 말아 주세요."

AI 답변은 그대로 정답이 아닙니다. **실제 실행 로그와 최소 3건의 주장·선택을 비교**하세요.


## 원본 교관용 코드 — 참고 구현

다음은 제공된 Solution의 해당 미션 코드입니다. 실제 수행 시 환경에 필요한 패키지를 확인하고 실행하세요.


In [ ]:
from pathlib import Path

if not (globals().get('MNT02_PROJECT_READY', False) and globals().get('MNT02_PROJECT_DIR') and Path(MNT02_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """야전 정비창 수리부속 청구/출고 이력과 장비 가동률 데이터의 업무 프로세스 맵, 데이터 스키마(피처 20개 이상)와 데이터 딕셔너리를 정의하고 가상 청구 기록을 검증한다."""
    import json
    import re
    import numpy as np
    import pandas as pd

    # 1) 업무 프로세스 맵: 단계별로 어떤 데이터가 생기는지 연결한다
    PROCESS = [("1. 정비 소요 발생", "장비 고장·계획정비", "equip_id, equip_type, maint_type"),
               ("2. 수리부속 청구", "정비반 -> 정비창", "req_id, req_date, nsn, qty_req, priority"),
               ("3. 재고 확인·불출", "정비창 재고 담당", "qty_issued, emergency_flag"),
               ("4. 조달 요청", "재고 부족 시 조달부서", "lead_time_m, supplier_count, unit_price"),
               ("5. 단종 위험 검토", "품목 관리 담당", "prod_status, alt_available"),
               ("6. 정비 완료·가동률 집계", "월 단위 집계", "op_hours, availability, downtime_h")]
    print("분석 목표: 월별 품목 수요를 예측해 결품·과잉재고를 줄이고, 단종예고·단종 품목의 최종 구매 수량 판단을 지원한다.")
    print("[업무 프로세스 맵] 정비 소요 -> 청구 -> 불출 -> 조달 -> 단종 검토 -> 가동률")
    for step, who, fields in PROCESS:
        print(f"  {step:18s} | {who:18s} | {fields}")

    # 2) 스키마(데이터 딕셔너리): 그룹, 자료형, 규칙, 설명
    NSN_RE = r"^\d{4}-\d{2}-\d{3}-\d{4}$"      # NSN = FSC(4) + 국가코드(2) + 일련번호(7), NIIN = 뒤 9자리
    SCHEMA = {
        "청구/불출": {"req_id": (str, {"pattern": r"^R\d{6}$"}, "청구 번호"),
                  "req_date": (str, {"pattern": r"^\d{4}-\d{2}-\d{2}$"}, "청구 일자"),
                  "unit_code": (str, {"pattern": r"^U-\d{2}$"}, "청구 부대(가상 코드)"),
                  "equip_id": (str, {"pattern": r"^[ABC]-\d{3}$"}, "장비 번호(가상)"),
                  "equip_type": (str, {"enum": ["A", "B", "C"]}, "장비 유형"),
                  "maint_type": (str, {"enum": ["계획", "비계획"]}, "정비 구분"),
                  "nsn": (str, {"pattern": NSN_RE}, "국가재고번호(가상)"),
                  "fsc": (str, {"pattern": r"^\d{4}$"}, "군수품 분류(NSN 앞 4자리)"),
                  "niin": (str, {"pattern": r"^\d{9}$"}, "품목 식별번호(NSN 뒤 9자리)"),
                  "qty_req": (int, {"min": 1, "max": 500}, "청구 수량"),
                  "qty_issued": (int, {"min": 0, "max": 500}, "불출 수량"),
                  "unit_of_issue": (str, {"enum": ["EA", "SE", "BX"]}, "불출 단위"),
                  "priority": (str, {"enum": ["긴급", "일반"]}, "청구 우선순위"),
                  "emergency_flag": (bool, {}, "긴급 불출 여부(이상치로 지우지 않음)")},
        "품목/공급망": {"unit_price": (int, {"min": 0}, "단가(원, 가상)"),
                   "lead_time_m": (int, {"min": 0, "max": 36}, "조달 리드타임(월)"),
                   "supplier_count": (int, {"min": 0, "max": 20}, "공급 가능 업체 수"),
                   "prod_status": (str, {"enum": ["생산중", "단종예고", "단종"]}, "생산(단종) 상태"),
                   "alt_available": (bool, {}, "대체품 존재 여부")},
        "가동률(월)": {"ym": (str, {"pattern": r"^\d{4}-\d{2}$"}, "집계 월"),
                   "op_hours": (float, {"min": 0, "max": 744}, "장비 운용시간"),
                   "availability": (float, {"min": 0, "max": 1}, "장비 가동률"),
                   "downtime_h": (float, {"min": 0, "max": 744}, "정비 대기·수리 시간"),
                   "exercise_flag": (int, {"enum": [0, 1]}, "대규모 훈련 월 여부")},
    }


    def check(name, v, typ, rule):
        if typ is float and isinstance(v, int) and not isinstance(v, bool):
            v = float(v)
        if not isinstance(v, typ) or (typ is not bool and isinstance(v, bool)):
            return [f"{name}: 자료형 오류({type(v).__name__})"]
        errs = []
        if "enum" in rule and v not in rule["enum"]:
            errs.append(f"{name}: 허용값 아님 '{v}'")
        if "min" in rule and v < rule["min"]:
            errs.append(f"{name}: 최소값 {rule['min']} 미만")
        if "max" in rule and v > rule["max"]:
            errs.append(f"{name}: 최대값 {rule['max']} 초과")
        if "pattern" in rule and not re.match(rule["pattern"], v):
            errs.append(f"{name}: 형식 불일치 '{v}'")
        return errs


    def validate(rec):
        errs = []
        for grp in SCHEMA.values():
            for k, (typ, rule, _) in grp.items():
                errs += [f"{k}: 필수 필드 누락"] if k not in rec else check(k, rec[k], typ, rule)
        if not errs:          # 필드 간 논리 규칙
            if rec["nsn"][:4] != rec["fsc"] or rec["nsn"].replace("-", "")[4:] != rec["niin"]:
                errs.append("nsn/fsc/niin 불일치")
            if rec["qty_issued"] > rec["qty_req"]:
                errs.append("불출 수량이 청구 수량 초과")
        return errs


    # 3) [교육용가상자료] 청구 기록 20건 생성
    rng = np.random.default_rng(1)
    records = []
    for i in range(20):
        fsc = str(rng.choice(["2520", "2530", "2940", "6140"]))
        niin = f"00{rng.integers(1000000, 9999999)}"
        q = int(rng.integers(1, 20))
        et = str(rng.choice(["A", "B", "C"]))
        records.append({"req_id": f"R{250001 + i}", "req_date": f"2025-{rng.integers(1, 13):02d}-{rng.integers(1, 29):02d}",
                        "unit_code": f"U-{rng.integers(1, 6):02d}", "equip_id": f"{et}-{rng.integers(1, 200):03d}", "equip_type": et,
                        "maint_type": str(rng.choice(["계획", "비계획"])), "nsn": f"{fsc}-{niin[:2]}-{niin[2:5]}-{niin[5:]}",
                        "fsc": fsc, "niin": niin, "qty_req": q, "qty_issued": int(rng.integers(0, q + 1)),
                        "unit_of_issue": "EA", "priority": str(rng.choice(["긴급", "일반"], p=[0.1, 0.9])),
                        "emergency_flag": bool(rng.random() < 0.05), "unit_price": int(rng.integers(10, 900)) * 1000,
                        "lead_time_m": int(rng.integers(1, 13)), "supplier_count": int(rng.integers(1, 5)),
                        "prod_status": str(rng.choice(["생산중", "단종예고", "단종"], p=[0.7, 0.2, 0.1])),
                        "alt_available": bool(rng.random() < 0.5), "ym": "2025-12", "op_hours": float(round(rng.uniform(80, 180), 1)),
                        "availability": float(round(rng.uniform(0.75, 0.97), 3)), "downtime_h": float(round(rng.uniform(5, 60), 1)),
                        "exercise_flag": int(rng.integers(0, 2))})

    ok = sum(1 for r in records if not validate(r))
    n_feat = sum(len(g) for g in SCHEMA.values())
    print(f"\n정의한 피처(필드) {n_feat}개 (목표 20개 이상: {'달성' if n_feat >= 20 else '미달'})")
    print(f"정상 샘플 {len(records)}건 중 검증 통과 {ok}건, 오류 {len(records) - ok}건")

    # 4) 오류 주입 시험
    bad = json.loads(json.dumps(records[0], ensure_ascii=False))
    bad["nsn"] = "2520-001234567"            # NSN 형식 오류
    bad["qty_req"] = -3                      # 음수 수량
    bad["prod_status"] = "미상"              # 허용값 아님
    del bad["lead_time_m"]                   # 필수 필드 누락
    bad["availability"] = 1.4                # 가동률 범위 초과
    print("[오류 주입 결과]")
    for e in validate(bad):
        print(" -", e)
    bad2 = json.loads(json.dumps(records[1], ensure_ascii=False)); bad2["fsc"] = "9999"
    print(" - (논리 규칙)", validate(bad2))

    # 5) 데이터 딕셔너리 출력
    rows = [(g, k, t.__name__, json.dumps(r, ensure_ascii=False) if r else "-", d) for g, grp in SCHEMA.items() for k, (t, r, d) in grp.items()]
    dic = pd.DataFrame(rows, columns=["그룹", "필드", "자료형", "규칙", "설명"])
    pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 34)
    print("[데이터 딕셔너리]")
    print(dic.to_string(index=False))


## 원본 교관용 예상 출력 (참고용)

아래 내용은 원본에 포함된 예시이며 **실제 실행 결과로 제출할 수 없습니다.**

<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
분석 목표: 월별 품목 수요를 예측해 결품·과잉재고를 줄이고, 단종예고·단종 품목의 최종 구매 수량 판단을 지원한다.
[업무 프로세스 맵] 정비 소요 -> 청구 -> 불출 -> 조달 -> 단종 검토 -> 가동률
  1. 정비 소요 발생        | 장비 고장·계획정비         | equip_id, equip_type, maint_type
  2. 수리부속 청구         | 정비반 -> 정비창         | req_id, req_date, nsn, qty_req, priority
  3. 재고 확인·불출        | 정비창 재고 담당          | qty_issued, emergency_flag
  4. 조달 요청           | 재고 부족 시 조달부서       | lead_time_m, supplier_count, unit_price
  5. 단종 위험 검토        | 품목 관리 담당           | prod_status, alt_available
  6. 정비 완료·가동률 집계    | 월 단위 집계            | op_hours, availability, downtime_h

정의한 피처(필드) 24개 (목표 20개 이상: 달성)
정상 샘플 20건 중 검증 통과 20건, 오류 0건
[오류 주입 결과]
 - nsn: 형식 불일치 '2520-001234567'
 - qty_req: 최소값 1 미만
 - lead_time_m: 필수 필드 누락
 - prod_status: 허용값 아님 '미상'
 - availability: 최대값 1 초과
 - (논리 규칙) ['nsn/fsc/niin 불일치']
[데이터 딕셔너리]
    그룹             필드   자료형                                           규칙                    설명
 청구/불출         req_id   str                     {"pattern": "^R\\d{6}$"}                 청구 번호
 청구/불출       req_date   str        {"pattern": "^\\d{4}-\\d{2}-\\d{2}$"}                 청구 일자
 청구/불출      unit_code   str                    {"pattern": "^U-\\d{2}$"}          청구 부대(가상 코드)
 청구/불출       equip_id   str                {"pattern": "^[ABC]-\\d{3}$"}             장비 번호(가상)
 청구/불출     equip_type   str                    {"enum": ["A", "B", "C"]}                 장비 유형
 청구/불출     maint_type   str                      {"enum": ["계획", "비계획"]}                 정비 구분
 청구/불출            nsn   str {"pattern": "^\\d{4}-\\d{2}-\\d{3}-\\d{4}$"}            국가재고번호(가상)
 청구/불출            fsc   str                      {"pattern": "^\\d{4}$"}     군수품 분류(NSN 앞 4자리)
 청구/불출           niin   str                      {"pattern": "^\\d{9}$"}    품목 식별번호(NSN 뒤 9자리)
 청구/불출        qty_req   int                       {"min": 1, "max": 500}                 청구 수량
 청구/불출     qty_issued   int                       {"min": 0, "max": 500}                 불출 수량
 청구/불출  unit_of_issue   str                 {"enum": ["EA", "SE", "BX"]}                 불출 단위
 청구/불출       priority   str                       {"enum": ["긴급", "일반"]}               청구 우선순위
 청구/불출 emergency_flag  bool                                            - 긴급 불출 여부(이상치로 지우지 않음)
품목/공급망     unit_price   int                                   {"min": 0}             단가(원, 가상)
품목/공급망    lead_time_m   int                        {"min": 0, "max": 36}            조달 리드타임(월)
품목/공급망 supplier_count   int                        {"min": 0, "max": 20}            공급 가능 업체 수
품목/공급망    prod_status   str              {"enum": ["생산중", "단종예고", "단종"]}             생산(단종) 상태
품목/공급망  alt_available  bool                                            -             대체품 존재 여부
가동률(월)             ym   str               {"pattern": "^\\d{4}-\\d{2}$"}                  집계 월
... (이하 생략)
```
</details>


## 교관 검토 기록

학생의 실제 실행 로그·대화 원문·수정 이력·평가 기준 충족 여부를 확인하고, 달성/미달/추가확인 및 승인 여부를 별도 기록합니다.

**통합 입력·저장 기능:** 마지막 통합 화면에서 미션별 산출물을 파일별로 작성·수정·저장하고, 실제 AI 질문·답변 원문, 원자료·출력 대조 근거, 검토 3건, 수정과 최종 수행 상태를 기록한 뒤 ZIP으로 다운로드합니다.

저장만으로 PASS나 승인이 되지 않습니다.


In [ ]:
# @title ⑤ 교관용 개별 산출물·검토기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('MNT02_PROJECT_READY', False) and globals().get('MNT02_PROJECT_DIR') and Path(MNT02_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 1
        DELIVERABLE_PATHS = ['docs/mnt02_logistics_domain_spec.md', 'forms/MNT02_M01_forms.md', 'reports/mnt02_data_dictionary.md', 'src/domain/mnt02_domain.py']
        PROJECT_NAME = 'MNT02'
        NOTEBOOK_ROLE = '교관용 Solution'
        WORK_ROOT = Path(MNT02_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'MNT02_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'MNT02_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                state['dirty'] = change['new'] != state['saved_value']
                state['saved'] = state['loaded'] and not state['dirty']
                if state['dirty']:
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
                elif state['saved']:
                    state['status'].value = '<span style="color:#286">저장된 내용과 같습니다.</span>'
                else:
                    state['status'].value = '<span style="color:#777">아직 저장하지 않았습니다.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','실습 계획·검토 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE, 'notebook_role': NOTEBOOK_ROLE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# MNT02 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('실습 계획·검토 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if record_dirty['value'] or any(v['dirty'] for v in editors.values()):
                    print('미저장 변경이 있습니다. 먼저 저장 후 ZIP을 다시 만드세요.')
                    return
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ MNT02 미션 {MISSION_CODE:02d} ({NOTEBOOK_ROLE}) 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 교관용 해설 및 예상 출력은 실제 실행·승인 증거가 아닙니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
